# Ejercicio 2 — Ascenso por máxima pendiente partiendo de un $3^2$

**Objetivo.** Usar el diseño $3^2$ para ajustar un modelo de primer orden (solo términos
lineales), calcular la dirección de ascenso por máxima pendiente y construir la trayectoria
de exploración hacia el óptimo.

**Factores:**
- $A$ = pH del medio de fermentación: 5.5 (−1), 6.5 (0), 7.5 (+1)
- $B$ = Temperatura: 28 (−1), 32 (0), 36 °C (+1)

**Respuesta:** Biomasa producida (g/L)

**Dataset:** `../../datos/fermentacion-3k.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = pd.read_csv('../../datos/fermentacion-3k.csv')
print(df)

## 1. Verificación de curvatura

Antes de lanzar el ascenso, verificamos si hay curvatura en la región actual ajustando el
modelo completo de segundo orden y evaluando los términos cuadráticos.

Si los p-valores de $x_1^2$ y $x_2^2$ son **mayores que 0.05**, no hay curvatura significativa
y el modelo de primer orden es una descripción adecuada de la superficie local: podemos
calcular la dirección de ascenso por máxima pendiente con confianza.

> **¿Qué pasa si la curvatura es significativa?**  
> Significa que probablemente ya estamos cerca del óptimo. En ese caso, se omite el ascenso
> y se augmenta el diseño a un CCD para ajustar el modelo de segundo orden directamente.

In [ ]:
# Modelo completo de 2° orden para verificar curvatura
modelo_2o = smf.ols('biomasa ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2', data=df).fit()
anova_2o = sm.stats.anova_lm(modelo_2o, typ=1)
print('ANOVA modelo cuadrático:')
print(anova_2o.round(4))

# La curvatura se evalúa con los términos cuadráticos; la interacción es un efecto distinto
p_curv_A = anova_2o.loc['I(x1 ** 2)', 'PR(>F)']
p_curv_B = anova_2o.loc['I(x2 ** 2)', 'PR(>F)']
print(f'\np-valor curvatura A: {p_curv_A:.4f}')
print(f'p-valor curvatura B: {p_curv_B:.4f}')

## 2. Modelo de primer orden

Ajustamos el modelo $y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \varepsilon$.

In [ ]:
modelo_1o = smf.ols('biomasa ~ x1 + x2', data=df).fit()
print(modelo_1o.summary())

b0 = modelo_1o.params['Intercept']
b1 = modelo_1o.params['x1']
b2 = modelo_1o.params['x2']
print(f'\nβ₀={b0:.3f},  β₁(pH)={b1:.3f},  β₂(Temp)={b2:.3f}')

## 3. Dirección de ascenso por máxima pendiente

La dirección del gradiente en variables codificadas es $\nabla\hat{y} = (\hat\beta_1, \hat\beta_2)$.
Normalizamos y calculamos el paso en unidades reales.

In [ ]:
# Gradiente normalizado
grad = np.array([b1, b2])
grad_norm = grad / np.linalg.norm(grad)
print(f'Gradiente (cod):       ({b1:.3f}, {b2:.3f})')
print(f'Gradiente normalizado: ({grad_norm[0]:.3f}, {grad_norm[1]:.3f})')

# Escalas reales: A: centro=6.5, delta=1; B: centro=32, delta=4
centro_real = np.array([6.5, 32.0])
delta_real  = np.array([1.0,  4.0])  # 1 unidad codificada = delta_real unidades reales

# Tabla de trayectoria de ascenso
paso_base = 0.3  # tamaño de paso a lo largo del vector gradiente normalizado
pasos = np.arange(0, 6)
print('\nTrayectoria de ascenso por máxima pendiente:')
print(f'{"Paso":>5} {"x1(pH cod)":>12} {"x2(T cod)":>12} {"pH_real":>10} {"T_real":>10} {"ŷ":>8}')
for s in pasos:
    x_cod = grad_norm * s * paso_base
    x_real = centro_real + x_cod * delta_real
    y_hat = b0 + b1*x_cod[0] + b2*x_cod[1]
    print(f'{s:>5} {x_cod[0]:>12.3f} {x_cod[1]:>12.3f} {x_real[0]:>10.2f} {x_real[1]:>10.2f} {y_hat:>8.2f}')

## 4. Visualización de la trayectoria

In [ ]:
x1g, x2g = np.meshgrid(np.linspace(-1.5, 2.5, 60), np.linspace(-1.5, 2.5, 60))
zg = b0 + b1*x1g + b2*x2g

fig, ax = plt.subplots(figsize=(7, 6))
cp = ax.contourf(x1g, x2g, zg, levels=15, cmap='YlGn')
plt.colorbar(cp, ax=ax, label='Biomasa (g/L) — modelo 1er orden')

# Datos observados
ax.scatter(df['x1'], df['x2'], c='black', zorder=5, label='Corridas $3^2$')

# Trayectoria de ascenso
traj = np.array([grad_norm * s * paso_base for s in pasos])
ax.plot(traj[:,0], traj[:,1], 'r-o', linewidth=2, markersize=7, label='Trayectoria de ascenso')

ax.set_xlabel('$x_1$ (pH)', fontsize=12)
ax.set_ylabel('$x_2$ (Temperatura)', fontsize=12)
ax.set_title('Ascenso por máxima pendiente desde un $3^2$')
ax.legend()
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
plt.tight_layout()
plt.show()

## 5. Conclusión

- El primer paso en RSM es **verificar curvatura**: si los términos cuadráticos no son
  significativos (como en este ejercicio), el modelo de primer orden es válido en la
  región actual y se puede calcular la dirección de ascenso.
- La dirección de ascenso se calcula como el gradiente $\nabla\hat{y} = (\hat\beta_1, \hat\beta_2)$,
  normalizado para avanzar en la dirección de mayor pendiente.
- La tabla de la trayectoria guía al experimentador hacia regiones de mayor rendimiento.
- Cuando la respuesta deja de aumentar a lo largo de la trayectoria, se centra un nuevo
  diseño (CCD o $3^2$) en ese punto para ajustar el modelo de segundo orden y localizar
  el óptimo.